# County Rainfall and Planting Advisory

This notebook cleans rainfall data from five Kenyan counties, analyses rainfall patterns using NumPy and produces planting-window recommendations for a farmer-support NGO.

In [ ]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## Task 1: Clean the Machakos Data

I will convert valid rainfall values to integers using a function with try and except. Invalid values will be replaced with the rounded average of the valid Machakos rainfall values, and I will print the months that were imputed.

In [ ]:
months = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

def clean_rainfall(values):
    """Convert valid rainfall values to integers and return cleaned data with imputed months."""
    cleaned = []
    invalid_positions = []

    for i, value in enumerate(values):
        try:
            cleaned.append(int(value.strip()))
        except ValueError:
            cleaned.append(None)
            invalid_positions.append(i)

    valid_values = [value for value in cleaned if value is not None]
    average_value = round(sum(valid_values) / len(valid_values))

    for i in invalid_positions:
        cleaned[i] = average_value

    return cleaned, invalid_positions


machakos_clean, imputed_positions = clean_rainfall(machakos_raw)

print("Cleaned Machakos rainfall:", machakos_clean)
print("Imputed months:")

for i in imputed_positions:
    print(months[i])

Cleaned Machakos rainfall: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed months:
April
September


## Task 2: Combine the County Rainfall Data

I will combine the four clean county rows with the cleaned Machakos row into one NumPy array. I will then confirm that the array has 5 rows, 12 columns and an integer data type.

In [ ]:
import numpy as np

R = np.array(clean_rows + [machakos_clean])

print("Shape:", R.shape)
print("Data type:", R.dtype)
print(R)

Shape: (5, 12)
Data type: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## Task 3: Annual Totals and Monthly Averages

I will use NumPy to calculate the total rainfall for each county and the average rainfall for each month across all five counties.

In [ ]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage rainfall by month:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual rainfall totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall by month:
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## Task 4: Wettest Month for Each County

I will use NumPy's argmax function with axis=1 to find the position of the wettest month for each county, then match each position to the correct month name.

In [ ]:
wettest_positions = np.argmax(R, axis=1)
wettest_months = [months[i] for i in wettest_positions]

for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")

Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## Task 5: Count Dry Months

I will use a NumPy boolean mask to identify months with rainfall below 50 mm, then count the number of dry months for each county.

In [ ]:
dry_months = (R < 50).sum(axis=1)

for county, count in zip(counties, dry_months):
    print(f"{county}: {count} dry months")

Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


## Task 6: Recommended Planting Window

I will add each month to the month that follows it, then find the pair of consecutive months with the highest combined rainfall for each county. This pair will be used as the recommended planting window.

In [ ]:
two_month_totals = R[:, :-1] + R[:, 1:]

best_window_positions = np.argmax(two_month_totals, axis=1)

planting_windows = [
    f"{months[i]}-{months[i + 1]}"
    for i in best_window_positions
]

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")

Nairobi: April-May
Nakuru: April-May
Kisumu: April-May
Mombasa: April-May
Machakos: November-December


## Task 7: Create the Rainfall Summary CSV

I will write the final county results to a CSV file using Python's csv module. The file will contain one row for each county.

In [ ]:
import csv

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "county",
            "annual_total_mm",
            "wettest_month",
            "dry_months",
            "planting_window"
        ]
    )

    writer.writeheader()

    for county, total, wettest, dry, window in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_months,
        planting_windows
    ):
        writer.writerow({
            "county": county,
            "annual_total_mm": int(total),
            "wettest_month": wettest,
            "dry_months": int(dry),
            "planting_window": window
        })

print("rainfall_summary.csv created successfully")

rainfall_summary.csv created successfully


## Planting Advisory

Kisumu received the highest annual rainfall at **1,371 mm**, while Machakos received the lowest at **811 mm**. Nairobi, Nakuru, Kisumu and Mombasa have their strongest two-month rainfall period in **April-May**, so this is the recommended planting window for those counties. Machakos has its strongest two-month period in **November-December**, so planting advice there should focus on that later window. Machakos also has **6 dry months**, while Kisumu has **0 dry months**, showing that rainfall reliability differs greatly between counties. The NGO should therefore send county-specific planting advice instead of using one planting period for all five counties.

## Data Quality Notes

The original Machakos data contained invalid values for **April** and **September**. These values were replaced with the rounded average of the valid Machakos rainfall readings, which was **68 mm**. Because these two months were estimated rather than directly measured, conclusions involving Machakos should be interpreted with some caution.

In [ ]:
%%writefile requirements.txt
numpy

Overwriting requirements.txt


In [ ]:
!git clone https://github.com/robyke/plp-da-a2-rainfall-advisory.git

Cloning into 'plp-da-a2-rainfall-advisory'...


In [ ]:
!cp rainfall_summary.csv plp-da-a2-rainfall-advisory/
!cp requirements.txt plp-da-a2-rainfall-advisory/